# 04. Datezo - Model Evaluation & Explainability

This notebook evaluates the serialized Datezo production model on the held-out test set, performs SHAP feature importance analysis, and tests the end-to-end inference prediction interface.

In [1]:
import os
import sys
import json
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

from src.evaluate import evaluate_production_model
from src.predict import predict_match

print("Executing production model evaluation...")
report = evaluate_production_model()

Executing production model evaluation...


## Classification Metrics & Confusion Matrix

In [2]:
print("Held-out Test Metrics:")
for k, v in report["classification_metrics"].items():
    print(f"  {k:<15}: {v}")

print("\nConfusion Matrix Breakdown:")
for k, v in report["confusion_matrix"].items():
    if k != "interpretation":
        print(f"  {k:<15}: {v}")

Held-out Test Metrics:
  accuracy       : 0.7623
  precision      : 0.4066
  recall         : 0.3501
  f1_score       : 0.3762
  roc_auc        : 0.7033
  pr_auc         : 0.4095
  log_loss       : 0.4608
  brier_score    : 0.147

Confusion Matrix Breakdown:
  true_positive  : 209
  true_negative  : 2013
  false_positive : 305
  false_negative : 388


## SHAP Feature Importances

In [3]:
feat_imp_path = os.path.join("..", "reports", "feature_importance.csv")
if not os.path.exists(feat_imp_path):
    feat_imp_path = os.path.join("reports", "feature_importance.csv")

df_imp = pd.read_csv(feat_imp_path)
print("Top 10 Most Important Features:")
display(df_imp.head(10))

Top 10 Most Important Features:


NameError: name 'display' is not defined

## End-to-End Prediction Interface Demo

In [4]:
sample_pair = {
    "male_age": 28,
    "female_age": 27,
    "same_race": 1,
    "same_field": 0,
    "shared_interests": 8.4,
    "attr_of_female": 8.0,
    "sinc_of_female": 7.5,
    "intel_of_female": 8.2,
    "fun_of_female": 8.0,
    "amb_of_female": 7.1,
    "attr_of_male": 8.1,
    "sinc_of_male": 7.8,
    "intel_of_male": 7.9,
    "fun_of_male": 8.3,
    "amb_of_male": 7.5,
    "male_pref_attr": 30.0,
    "male_pref_intel": 25.0,
    "female_pref_attr": 28.0,
    "female_pref_intel": 27.0,
    "male_self_attr": 8.0,
    "female_self_attr": 7.8,
    "male_goes_out": "often",
    "female_goes_out": "often"
}

res = predict_match(sample_pair)
print(json.dumps(res, indent=2))

{
  "prediction": 1,
  "label": "MATCH",
  "match_probability": 79.1,
  "compatibility_category": "High Compatibility",
  "compatibility_index": 77.8,
  "positive_factors": [
    "High shared interests score",
    "Strong mutual attractiveness rating",
    "High mutual fun perception",
    "High mutual intelligence perception"
  ],
  "negative_factors": [
    "No major compatibility drawbacks detected"
  ]
}
